# WeedDet v5 — Retrain Notebook
**AgriNav | Benny Merryman-Smith**

| Cell | Purpose |
|------|---------|
| 0 | Mount Drive + load weeddet_Latest |
| 1 | Extract dataset + 80/20 train/val split |
| 2 | Sanity check dataset |
| 3 | Train 12 epochs (train + val loss per epoch) |
| 4 | Plot loss curves |
| 5 | Visual check |
| 6 | mAP eval (AP@0.5, AP@0.75) |


In [ ]:
from google.colab import drive
import sys, os

drive.mount('/content/drive')
SCRIPT_DIR = '/content/drive/MyDrive/weeddet_v2_checkpoints'

script_file = os.path.join(SCRIPT_DIR, 'weeddet_Latest.py')
assert os.path.exists(script_file), (
    f'''Script not found: {script_file}
'
    'Make sure weeddet_Latest.py is in that Drive folder.'''
)
sys.path.insert(0, SCRIPT_DIR)
print(f'Done — {script_file} on path')

## Cell 1 — Extract Dataset + 80/20 Train/Val Split

In [ ]:
import torch, random, glob, zipfile, os
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
ZIP_PATH    = '/content/drive/MyDrive/weeddet_v2_checkpoints/Rice_Classification.v1i.voc.zip'
EXTRACT_DIR = '/content/dataset'
FLAT_ROOT   = '/content/rice_flat'
CKPT_DIR    = '/content/drive/MyDrive/weeddet_v5_checkpoints'
SPLIT_SEED  = 42
VAL_RATIO   = 0.20

os.makedirs(CKPT_DIR, exist_ok=True)

# ── Unzip ──────────────────────────────────────────────────────────────────────
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
    print('Extracting dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('Dataset already extracted.')

# ── Build flat directory ───────────────────────────────────────────────────────
os.makedirs(f'{FLAT_ROOT}/images',      exist_ok=True)
os.makedirs(f'{FLAT_ROOT}/annotations', exist_ok=True)

all_stems = []
for split in ['train', 'valid', 'val']:  # 'test' excluded — kept as true holdout
    base = Path(f'{EXTRACT_DIR}/{split}')
    if not base.exists(): continue
    img_dir = base / 'images' if (base / 'images').exists() else base
    ann_dir = base / 'annotations' if (base / 'annotations').exists() else (
              base / 'labels'      if (base / 'labels').exists()      else base)
    for img_p in sorted(img_dir.glob('*')):
        if img_p.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
        xml_p = ann_dir / (img_p.stem + '.xml')
        if not xml_p.exists(): continue
        dst_i = Path(FLAT_ROOT) / 'images'      / img_p.name
        dst_x = Path(FLAT_ROOT) / 'annotations' / (img_p.stem + '.xml')
        if not dst_i.exists(): os.symlink(img_p.resolve(), dst_i)
        if not dst_x.exists(): os.symlink(xml_p.resolve(), dst_x)
        all_stems.append(img_p.stem)

# ── 80/20 split ────────────────────────────────────────────────────────────────
random.seed(SPLIT_SEED)
random.shuffle(all_stems)
cut         = int(len(all_stems) * (1 - VAL_RATIO))
train_stems = all_stems[:cut]
val_stems   = all_stems[cut:]

Path(f'{FLAT_ROOT}/train.txt').write_text('\n'.join(train_stems))
Path(f'{FLAT_ROOT}/val.txt'  ).write_text('\n'.join(val_stems))

print(f'Total  : {len(all_stems)} image+annotation pairs')
print(f'Train  : {len(train_stems)}')
print(f'Val    : {len(val_stems)}')
print(f'Ckpts  : {CKPT_DIR}')

## Cell 2 — Dataset Sanity Check

In [ ]:
for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
import weeddet_Latest as wd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_ds = wd.WeedDataset(FLAT_ROOT, 'train', img_size=512, augment=False)
val_ds   = wd.WeedDataset(FLAT_ROOT, 'val',   img_size=512, augment=False)

sample = next((train_ds[i] for i in range(min(20, len(train_ds))) if train_ds[i] is not None), None)
assert sample is not None, 'All samples returned None — check symlinks in FLAT_ROOT'
img_t, tgt = sample
assert img_t.shape == (3, 512, 512), f'Wrong shape {img_t.shape}'
print(f'Train  : {len(train_ds)} samples  |  shape {img_t.shape}  |  {len(tgt["boxes"])} GT boxes in sample  ✓')

val_sample = next((val_ds[i] for i in range(min(20, len(val_ds))) if val_ds[i] is not None), None)
assert val_sample is not None, 'All val samples returned None'
print(f'Val    : {len(val_ds)} samples  ✓')


## Cell 3 — Train 12 Epochs with Val Loss
Saves  (best val loss) and  every 4 epochs.


In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
import os # Added for os.path.exists

# Assuming CKPT_DIR is defined in a preceding cell or globally accessible
BEST_CKPT = f'{CKPT_DIR}/weeddet_v5_best.pth'

# ── Hyperparameters ────────────────────────────────────────────────────────────
NUM_EPOCHS        = 24
BATCH_SIZE        = 2
BASE_LR           = 0.001
MIN_LR            = 0.00005
MOMENTUM          = 0.9
WEIGHT_DECAY      = 0.0001
GRAD_CLIP         = 0.5
WARMUP_ITERS      = 200
WARMUP_FACTOR     = 0.001
ANCHOR_BASE_SCALE = 6
LSC_K             = 7
IMG_SIZE          = 512
NUM_CLASSES       = 1
SAVE_EVERY        = 4

def collate(batch):
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch)) if batch else ([], [])

train_loader = DataLoader(
    wd.WeedDataset(FLAT_ROOT, 'train', img_size=IMG_SIZE, augment=True),
    batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    wd.WeedDataset(FLAT_ROOT, 'val', img_size=IMG_SIZE, augment=False),
    batch_size=1, shuffle=False,
    collate_fn=collate, num_workers=2
)

model = wd.WeedDet(
    num_classes=NUM_CLASSES,
    anchor_base_scale=ANCHOR_BASE_SCALE,
    lsc_k=LSC_K,
).to(device)

# Freeze BN
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        for p in m.parameters(): p.requires_grad = False
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M total  |  '
      f'{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M trainable')

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=BASE_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
)

# ── Resume / Initialize training state ─────────────────────────────────────────
RESUME_TRAINING = True # Set to False to start training from scratch

start_epoch = 1
best_val_loss = float('inf')
global_step   = 0

if RESUME_TRAINING and os.path.exists(BEST_CKPT):
    print(f'Loading checkpoint from {BEST_CKPT} to resume training...')
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['loss']
    global_step = (start_epoch - 1) * len(train_loader) # Approximate global_step for scheduler
    print(f'Resuming from epoch {start_epoch-1} with best val loss {best_val_loss:.4f}.')
else:
    print('Starting new training session.')

# Initialize scheduler (re-initialize if resuming, as scheduler state is not saved in checkpoint)
# Calculate total_steps for the scheduler based on the remaining epochs
remaining_epochs = max(0, NUM_EPOCHS - (start_epoch - 1))
total_steps_for_scheduler = remaining_epochs * len(train_loader)

scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, total_steps_for_scheduler - WARMUP_ITERS), eta_min=MIN_LR
)

# ── Training loop ──────────────────────────────────────────────────────────────
train_losses, val_losses = [], [] # Reset losses for the current training run

print(f'''
Training {NUM_EPOCHS - (start_epoch-1)} additional epochs (total target {NUM_EPOCHS}) | {len(train_loader)} batches/epoch | device={device}
''')

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    # -- Train --
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()

    epoch_loss, n_batches = 0.0, 0
    for imgs, targets in train_loader:
        if not imgs: continue
        imgs    = torch.stack([i.to(device) for i in imgs])
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

        # Warmup LR
        if global_step < WARMUP_ITERS:
            warmup_lr = BASE_LR * (WARMUP_FACTOR + (1 - WARMUP_FACTOR) * global_step / WARMUP_ITERS)
            for pg in optimizer.param_groups: pg['lr'] = warmup_lr

        loss_dict = model(imgs, targets)
        loss      = loss_dict["total_loss"]  # FIX: was sum(values()) = 2×total

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=GRAD_CLIP
        )
        optimizer.step()

        if global_step >= WARMUP_ITERS:
            scheduler.step()

        epoch_loss += loss.item()
        n_batches  += 1
        global_step += 1

    avg_train = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_train)

    # -- Validate --
    model.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            if not imgs: continue
            imgs    = torch.stack([i.to(device) for i in imgs])
            targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
            # WeedDet returns loss dict in train mode; switch temporarily
            model.train()
            for m in model.modules():
                if isinstance(m, nn.BatchNorm2d): m.eval()
            loss_dict = model(imgs, targets)
            model.eval()
            val_loss += loss_dict["total_loss"].item()  # FIX: was sum(values()) = 2×total
            n_val    += 1

    avg_val = val_loss / max(n_val, 1)
    val_losses.append(avg_val)

    lr_now = optimizer.param_groups[0]['lr']
    star   = ''
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({'epoch': epoch, 'loss': avg_val,
                    'state_dict': model.state_dict(),
                    'optimizer': optimizer.state_dict()},
                   f'{CKPT_DIR}/weeddet_v5_best.pth')
        star = '  ★ Best checkpoint saved'

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  '
          f'train={avg_train:.4f}  val={avg_val:.4f}  '
          f'best={best_val_loss:.4f}  lr={lr_now:.6f}{star}')

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch': epoch, 'loss': avg_val,
                    'state_dict': model.state_dict(),
                    'optimizer': optimizer.state_dict()},
                   f'{CKPT_DIR}/weeddet_v5_epoch{epoch}.pth') # Also save optimizer state in epoch checkpoints
        print(f'  Checkpoint saved -> weeddet_v5_epoch{epoch}.pth')

print(f'''
Training complete. Best val loss: {best_val_loss:.4f}''')
print(f'Best checkpoint: {CKPT_DIR}/weeddet_v5_best.pth')

## Cell 4 — Loss Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, len(train_losses) + 1))
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_losses, marker='o', label='Train loss', color='steelblue')
ax.plot(epochs, val_losses,   marker='s', label='Val loss',   color='tomato')
best_ep = val_losses.index(min(val_losses)) + 1
ax.axvline(best_ep, color='tomato', linestyle='--', alpha=0.5,
           label=f'Best val (epoch {best_ep}, loss={min(val_losses):.4f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('WeedDet v5 — Train vs Val Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/loss_curve.png', dpi=150)
plt.show()
print(f'Saved to {CKPT_DIR}/loss_curve.png')


## Cell 5 — Visual Check
Loads  and runs inference on a random val image.  
**Green** = ground truth | **Red** = predictions


In [ ]:
import glob, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision.transforms as T
from PIL import Image

for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
import weeddet_Latest as wd

BEST_CKPT = f'{CKPT_DIR}/weeddet_v5_best.pth'
SCORE_THR = 0.05
NMS_THR   = 0.25

ckpt = torch.load(BEST_CKPT, map_location=device)
vis_model = wd.WeedDet(num_classes=1, anchor_base_scale=6, lsc_k=7).to(device)
vis_model.load_state_dict(ckpt['state_dict'])
vis_model.eval()
print(f'Loaded epoch {ckpt["epoch"]}  val_loss={ckpt["loss"]:.4f}')

# Pick a random val image
val_stems_list = Path(f'{FLAT_ROOT}/val.txt').read_text().splitlines()
stem = random.choice(val_stems_list)
img_path = next(iter(
    glob.glob(f'{FLAT_ROOT}/images/{stem}.*')), None)
assert img_path, f'Image not found for stem: {stem}'

img_orig = Image.open(img_path).convert('RGB')
img_lb, scale, pl, pt = wd.letterbox_pil(img_orig, 512)
tf = T.Compose([T.ToTensor(),
                T.Normalize(mean=wd.IMAGENET_MEAN, std=wd.IMAGENET_STD)])
tensor = tf(img_lb).unsqueeze(0).to(device)

with torch.no_grad():
    cls_l, regs, anchors, ishape = vis_model._get_logits(tensor)
    results = vis_model._decode(cls_l, regs, anchors, ishape,
                                score_thr=0.05, nms_thr=NMS_THR,
                                max_dets=300, output_thr=SCORE_THR)

pred_boxes  = wd.unpad_boxes(results[0]['boxes'].cpu(), scale, pl, pt)
pred_scores = results[0]['scores'].cpu()

# Load GT boxes from VOC XML
import xml.etree.ElementTree as ET
xml_path = f'{FLAT_ROOT}/annotations/{stem}.xml'
gt_boxes = []
if os.path.exists(xml_path):
    tree = ET.parse(xml_path)
    for obj in tree.findall('object'):
        bb = obj.find('bndbox')
        gt_boxes.append([int(bb.find(t).text) for t in ['xmin','ymin','xmax','ymax']])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
img_np = np.array(img_orig)
for ax, title, boxes, scores, color in [
    (axes[0], f'Ground truth ({len(gt_boxes)} boxes)', gt_boxes, None, 'lime'),
    (axes[1], f'v5 Predictions (score>={SCORE_THR}, {len(pred_boxes)} boxes)',
     pred_boxes.tolist(), pred_scores.tolist(), 'red'),
]:
    ax.imshow(img_np); ax.axis('off'); ax.set_title(title, fontsize=11)
    for i, b in enumerate(boxes):
        x1,y1,x2,y2 = b
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                     linewidth=2, edgecolor=color, facecolor='none'))
        label = f'rice {scores[i]:.2f}' if scores else 'rice (gt)'
        ax.text(x1, max(y1-4,0), label, color=color,
                fontsize=8, backgroundcolor='black')
plt.tight_layout(); plt.show()
print(f'GT: {len(gt_boxes)}  Pred: {len(pred_boxes)}')
if len(pred_scores): print(f'Score range: {pred_scores.min():.3f}-{pred_scores.max():.3f}')


## Cell 6 — mAP Evaluation (AP@0.5 and AP@0.75)
Runs inference on the full val set and computes COCO-style mAP using pycocotools.


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'pycocotools', '-q'])
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import xml.etree.ElementTree as ET
import json, glob

for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
import weeddet_Latest as wd

BEST_CKPT = f'{CKPT_DIR}/weeddet_v5_best.pth'
SCORE_THR = 0.001  # FIXED: must be near-zero for valid COCO mAP recall
NMS_THR   = 0.45   # FIXED: 0.25 was too aggressive for dense rice scenes

ckpt = torch.load(BEST_CKPT, map_location=device)
eval_model = wd.WeedDet(num_classes=1, anchor_base_scale=6, lsc_k=7).to(device)
eval_model.load_state_dict(ckpt['state_dict'])
eval_model.eval()

val_stems_list = Path(f'{FLAT_ROOT}/val.txt').read_text().splitlines()
print(f'Evaluating on {len(val_stems_list)} val images...')

# ── Build COCO ground-truth ────────────────────────────────────────────────────
coco_gt = {'images': [], 'annotations': [], 'categories': [{'id':1,'name':'rice'}]}
ann_id = 1
for img_id, stem in enumerate(val_stems_list, 1):
    img_path = next(iter(glob.glob(f'{FLAT_ROOT}/images/{stem}.*')), None)
    if not img_path: continue
    pil = Image.open(img_path).convert('RGB')
    coco_gt['images'].append({'id': img_id, 'width': pil.width, 'height': pil.height})
    xml_path = f'{FLAT_ROOT}/annotations/{stem}.xml'
    if not os.path.exists(xml_path): continue
    for obj in ET.parse(xml_path).findall('object'):
        bb = obj.find('bndbox')
        x1,y1,x2,y2 = [int(bb.find(t).text) for t in ['xmin','ymin','xmax','ymax']]
        coco_gt['annotations'].append({
            'id': ann_id, 'image_id': img_id, 'category_id': 1,
            'bbox': [x1, y1, x2-x1, y2-y1], 'area': (x2-x1)*(y2-y1), 'iscrowd': 0
        })
        ann_id += 1

gt_json = '/tmp/coco_gt.json'
with open(gt_json, 'w') as f: json.dump(coco_gt, f)
coco_gt_api = COCO(gt_json)

# ── Run inference on val set ───────────────────────────────────────────────────
import torchvision.transforms as T
tf = T.Compose([T.ToTensor(),
                T.Normalize(mean=wd.IMAGENET_MEAN, std=wd.IMAGENET_STD)])

detections = []
for img_id, stem in enumerate(val_stems_list, 1):
    img_path = next(iter(glob.glob(f'{FLAT_ROOT}/images/{stem}.*')), None)
    if not img_path: continue
    img_orig = Image.open(img_path).convert('RGB')
    img_lb, scale, pl, pt = wd.letterbox_pil(img_orig, 512)
    tensor = tf(img_lb).unsqueeze(0).to(device)
    with torch.no_grad():
        cls_l, regs, anchors, ishape = eval_model._get_logits(tensor)
        results = eval_model._decode(cls_l, regs, anchors, ishape,
                                     score_thr=SCORE_THR, nms_thr=NMS_THR,
                                     max_dets=1000, output_thr=SCORE_THR)
    boxes  = wd.unpad_boxes(results[0]['boxes'].cpu(), scale, pl, pt)
    scores = results[0]['scores'].cpu()
    for box, score in zip(boxes.tolist(), scores.tolist()):
        x1,y1,x2,y2 = box
        detections.append({
            'image_id': img_id, 'category_id': 1,
            'bbox': [x1, y1, x2-x1, y2-y1], 'score': score
        })

if not detections:
    print('WARNING: No detections above threshold on val set. Check model or lower SCORE_THR.')
else:
    coco_dt = coco_gt_api.loadRes(detections)
    coco_eval = COCOeval(coco_gt_api, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    ap50  = coco_eval.stats[1]   # AP @ IoU=0.50
    ap75  = coco_eval.stats[2]   # AP @ IoU=0.75
    ap_all = coco_eval.stats[0]  # AP @ IoU=0.50:0.95

    print(f'''
--- WeedDet v5 mAP Results ---
''')
    print(f'AP@0.50:0.95 : {ap_all:.4f}')
    print(f'AP@0.50      : {ap50:.4f}')
    print(f'AP@0.75      : {ap75:.4f}')
    print(f'Val images   : {len(val_stems_list)}')
    print(f'Detections   : {len(detections)}')